# Week 2 — SQL-Based Data Analysis (ShopEase E-Commerce)
**Celebal Technologies — Summer Internship 2026 (Data Engineering Track)**

**Author:** Laxmi Mishra

## 1. Scenario & Business Context

You are a Junior Data Analyst at **ShopEase**, a mid-sized e-commerce company that sells electronics, clothing, and home products across India. The management team wants to understand sales patterns, customer behavior, and product performance to make better business decisions.

The database has 4 related tables: `customers`, `products`, `orders`, `order_items`. This notebook uses **SQLite (via Python's `sqlite3`)** as the SQL engine, which follows standard SQL syntax as required.

**Structure of this notebook:**
- Section 0 — Database setup, schema creation, data loading
- Section A — SQL Basics (SELECT, Constraints, Primary Keys)
- Section B — Filtering & Optimization (WHERE, Indexes)
- Section C — Aggregation (GROUP BY, SUM, COUNT, AVG, MIN, MAX)
- Section D — Joins & Relationships
- Section E — Advanced Concepts (CASE, ACID, Transactions)


## Section 0 — Database Setup

We create a SQLite database and load it with the helper libraries needed to run queries and display results as clean tables.

In [1]:
import sqlite3
import pandas as pd

# Create (or reset) the database file
conn = sqlite3.connect('shopease.db')
conn.execute("PRAGMA foreign_keys = ON;")  # enforce FK constraints (off by default in SQLite)
cursor = conn.cursor()

def run_query(query, params=None):
    """Run a SELECT query and return results as a pandas DataFrame."""
    return pd.read_sql_query(query, conn, params=params)

def run_sql(query):
    """Run a DDL/DML statement (CREATE, INSERT, UPDATE, etc.)"""
    cursor.execute(query)
    conn.commit()

print("Connected to shopease.db")


Connected to shopease.db


### 2. Database Schema & Constraints

Dropping old tables first (so the notebook can be re-run cleanly), then creating the 4 tables exactly as specified, including Primary Keys, Foreign Keys, CHECK constraints, and Indexes.

In [2]:
# Drop tables if they already exist (clean re-run)
for t in ['order_items', 'orders', 'products', 'customers']:
    cursor.execute(f"DROP TABLE IF EXISTS {t}")
conn.commit()

# ---------- customers ----------
run_sql('''
CREATE TABLE customers (
    customer_id   INTEGER       PRIMARY KEY,
    first_name    VARCHAR(50)   NOT NULL,
    last_name     VARCHAR(50)   NOT NULL,
    email         VARCHAR(100)  UNIQUE NOT NULL,
    city          VARCHAR(50)   NOT NULL,
    state         VARCHAR(50)   NOT NULL,
    join_date     DATE          NOT NULL,
    is_premium    BOOLEAN       DEFAULT 0
)
''')
run_sql("CREATE INDEX idx_customers_city ON customers(city)")
run_sql("CREATE INDEX idx_customers_state ON customers(state)")

# ---------- products ----------
run_sql('''
CREATE TABLE products (
    product_id    INTEGER       PRIMARY KEY,
    product_name  VARCHAR(100)  NOT NULL,
    category      VARCHAR(50)   NOT NULL,
    brand         VARCHAR(50)   NOT NULL,
    unit_price    DECIMAL(10,2) NOT NULL CHECK (unit_price > 0),
    stock_qty     INTEGER       NOT NULL DEFAULT 0 CHECK (stock_qty >= 0)
)
''')
run_sql("CREATE INDEX idx_products_category ON products(category)")

# ---------- orders ----------
run_sql('''
CREATE TABLE orders (
    order_id      INTEGER       PRIMARY KEY,
    customer_id   INTEGER       NOT NULL,
    order_date    DATE          NOT NULL,
    status        VARCHAR(20)   NOT NULL DEFAULT 'Pending'
                  CHECK (status IN ('Pending','Shipped','Delivered','Cancelled')),
    total_amount  DECIMAL(12,2) NOT NULL CHECK (total_amount >= 0),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
''')
run_sql("CREATE INDEX idx_orders_date ON orders(order_date)")
run_sql("CREATE INDEX idx_orders_status ON orders(status)")

# ---------- order_items ----------
run_sql('''
CREATE TABLE order_items (
    item_id       INTEGER       PRIMARY KEY,
    order_id      INTEGER       NOT NULL,
    product_id    INTEGER       NOT NULL,
    quantity      INTEGER       NOT NULL CHECK (quantity > 0),
    unit_price    DECIMAL(10,2) NOT NULL CHECK (unit_price > 0),
    discount_pct  DECIMAL(5,2)  DEFAULT 0 CHECK (discount_pct BETWEEN 0 AND 100),
    FOREIGN KEY (order_id)   REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
)
''')

print("All 4 tables and their indexes created successfully.")


All 4 tables and their indexes created successfully.


### 3. Sample Data — Loading INSERT Statements

Loading the exact sample data provided in the assignment doc.

In [3]:
# ========== INSERT: customers ==========
customers_data = [
    (101, 'Aarav',  'Sharma', 'aarav.s@email.com',  'Mumbai',    'Maharashtra', '2024-01-15', 1),
    (102, 'Priya',  'Patel',  'priya.p@email.com',  'Ahmedabad', 'Gujarat',     '2024-02-20', 0),
    (103, 'Rohan',  'Gupta',  'rohan.g@email.com',  'Delhi',     'Delhi',       '2024-03-10', 1),
    (104, 'Sneha',  'Reddy',  'sneha.r@email.com',  'Hyderabad', 'Telangana',   '2024-04-05', 0),
    (105, 'Vikram', 'Singh',  'vikram.s@email.com', 'Jaipur',    'Rajasthan',   '2024-05-12', 1),
    (106, 'Ananya', 'Iyer',   'ananya.i@email.com', 'Chennai',   'Tamil Nadu',  '2024-06-18', 0),
    (107, 'Karan',  'Mehta',  'karan.m@email.com',  'Pune',      'Maharashtra', '2024-07-22', 1),
    (108, 'Divya',  'Nair',   'divya.n@email.com',  'Kochi',     'Kerala',      '2024-08-30', 0),
]
cursor.executemany("INSERT INTO customers VALUES (?,?,?,?,?,?,?,?)", customers_data)

# ========== INSERT: products ==========
products_data = [
    (201, 'Wireless Earbuds',     'Electronics', 'BoAt',         1499.00, 250),
    (202, 'Cotton T-Shirt',       'Clothing',    'Levis',        799.00,  500),
    (203, 'Smart Watch',          'Electronics', 'Noise',        2999.00, 150),
    (204, 'Running Shoes',        'Clothing',    'Nike',         4599.00, 120),
    (205, 'Bluetooth Speaker',    'Electronics', 'JBL',          3499.00, 200),
    (206, 'Bedsheet Set',         'Home',        'Spaces',       1299.00, 300),
    (207, 'Laptop Stand',         'Electronics', 'AmazonBasics', 899.00,  180),
    (208, 'Cushion Covers (Set)', 'Home',        'HomeCenter',   599.00,  400),
]
cursor.executemany("INSERT INTO products VALUES (?,?,?,?,?,?)", products_data)

# ========== INSERT: orders ==========
orders_data = [
    (1001, 101, '2024-08-01', 'Delivered', 4498.00),
    (1002, 102, '2024-08-03', 'Delivered', 799.00),
    (1003, 103, '2024-08-05', 'Shipped',   7498.00),
    (1004, 101, '2024-08-10', 'Delivered', 3499.00),
    (1005, 104, '2024-08-12', 'Cancelled', 2999.00),
    (1006, 105, '2024-08-15', 'Delivered', 5898.00),
    (1007, 106, '2024-08-18', 'Pending',   1299.00),
    (1008, 103, '2024-08-20', 'Delivered', 899.00),
    (1009, 107, '2024-08-25', 'Shipped',   6098.00),
    (1010, 108, '2024-08-28', 'Delivered', 1598.00),
]
cursor.executemany("INSERT INTO orders VALUES (?,?,?,?,?)", orders_data)

# ========== INSERT: order_items ==========
order_items_data = [
    (5001, 1001, 201, 2, 1499.00, 0),
    (5002, 1001, 207, 1, 899.00,  10),
    (5003, 1002, 202, 1, 799.00,  0),
    (5004, 1003, 203, 1, 2999.00, 0),
    (5005, 1003, 204, 1, 4599.00, 5),
    (5006, 1004, 205, 1, 3499.00, 0),
    (5007, 1005, 203, 1, 2999.00, 0),
    (5008, 1006, 201, 1, 1499.00, 10),
    (5009, 1006, 204, 1, 4599.00, 5),
    (5010, 1007, 206, 1, 1299.00, 0),
    (5011, 1008, 207, 1, 899.00,  0),
    (5012, 1009, 205, 1, 3499.00, 0),
    (5013, 1009, 208, 2, 599.00,  15),
    (5014, 1010, 206, 1, 1299.00, 0),
    (5015, 1010, 208, 1, 599.00,  0),
]
cursor.executemany("INSERT INTO order_items VALUES (?,?,?,?,?,?)", order_items_data)

conn.commit()
print("Sample data loaded successfully into all 4 tables.")


Sample data loaded successfully into all 4 tables.


### 4. Validation — Row Counts

Quick sanity check to confirm all rows loaded correctly before moving to the actual questions.

In [4]:
for table in ['customers', 'products', 'orders', 'order_items']:
    count = run_query(f"SELECT COUNT(*) AS row_count FROM {table}").iloc[0,0]
    print(f"{table:15s} -> {count} rows")


customers       -> 8 rows
products        -> 8 rows
orders          -> 10 rows
order_items     -> 15 rows


---
## Section A — SQL Basics (SELECT, Constraints, Primary Keys)
These questions test understanding of basic data retrieval, table structure, and database constraints.


**Q1. Write a query to display all columns and rows from the customer's table.**

In [5]:
q1 = run_query("SELECT * FROM customers")
q1


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


**Q2. Retrieve only the first_name, last_name, and city of all customers.**

In [6]:
q2 = run_query("SELECT first_name, last_name, city FROM customers")
q2


,first_name,last_name,city
0,Aarav,Sharma,Mumbai
1,Priya,Patel,Ahmedabad
2,Rohan,Gupta,Delhi
3,Sneha,Reddy,Hyderabad
4,Vikram,Singh,Jaipur
5,Ananya,Iyer,Chennai
6,Karan,Mehta,Pune
7,Divya,Nair,Kochi


**Q3. List all unique categories available in the products table.**

In [7]:
q3 = run_query("SELECT DISTINCT category FROM products")
q3


,category
0,Clothing
1,Electronics
2,Home


**Q4. Identify the Primary Key of each table. Explain why a Primary Key must be unique and NOT NULL.**

| Table | Primary Key |
|---|---|
| customers | `customer_id` |
| products | `product_id` |
| orders | `order_id` |
| order_items | `item_id` |

**Why a Primary Key must be UNIQUE and NOT NULL:**
A Primary Key is the column that uniquely identifies every single row in a table. If two rows were allowed to share the same Primary Key value, the database (and any query, JOIN, or Foreign Key referencing that table) would have no reliable way to tell the two rows apart — updates or deletes meant for one row could silently affect the wrong one. It must also be NOT NULL because a NULL value represents "unknown" — and an unknown value can never guarantee uniqueness or correctly identify a specific record. Together, UNIQUE + NOT NULL guarantee that every row has one, and only one, dependable identity.


**Q5. What constraints are applied to the email column? What happens on a duplicate email?**

The `email` column has a **UNIQUE** constraint and a **NOT NULL** constraint (`VARCHAR(100) UNIQUE NOT NULL`).

If we try to insert a duplicate email, SQLite will reject the statement and raise an `IntegrityError` (`UNIQUE constraint failed: customers.email`) — the row will **not** be inserted. Demonstrated below:


In [8]:
try:
    cursor.execute(
        "INSERT INTO customers VALUES (109, 'Test', 'User', 'aarav.s@email.com', 'Mumbai', 'Maharashtra', '2024-09-01', 0)"
    )
    conn.commit()
except sqlite3.IntegrityError as e:
    print("INSERT FAILED as expected.")
    print("Error:", e)
    conn.rollback()


INSERT FAILED as expected.
Error: UNIQUE constraint failed: customers.email


**Q6. Try inserting a product with unit_price = -50. What happens and which constraint prevents it?**

The `products` table has `unit_price DECIMAL(10,2) NOT NULL CHECK (unit_price > 0)`. A negative price violates the **CHECK constraint**, so the insert is rejected.


In [9]:
try:
    cursor.execute(
        "INSERT INTO products VALUES (209, 'Broken Product', 'Electronics', 'TestBrand', -50, 10)"
    )
    conn.commit()
except sqlite3.IntegrityError as e:
    print("INSERT FAILED as expected.")
    print("Error:", e)
    conn.rollback()


INSERT FAILED as expected.
Error: CHECK constraint failed: unit_price > 0


---
## Section B — Filtering & Optimization (WHERE, Indexes)
These questions test the ability to filter data effectively and understand how indexes improve query performance.


**Q7. Retrieve all orders with status = 'Delivered'.**

In [10]:
q7 = run_query("SELECT * FROM orders WHERE status = 'Delivered'")
q7


,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498
1,1002,102,2024-08-03,Delivered,799
2,1004,101,2024-08-10,Delivered,3499
3,1006,105,2024-08-15,Delivered,5898
4,1008,103,2024-08-20,Delivered,899
5,1010,108,2024-08-28,Delivered,1598


**Q8. Find all products in the 'Electronics' category with a unit_price greater than ₹2000.**

In [11]:
q8 = run_query("""
SELECT * FROM products
WHERE category = 'Electronics' AND unit_price > 2000
""")
q8


,product_id,product_name,category,brand,unit_price,stock_qty
0,203,Smart Watch,Electronics,Noise,2999,150
1,205,Bluetooth Speaker,Electronics,JBL,3499,200


**Q9. List all customers who joined in the year 2024 and belong to the state 'Maharashtra'.**

In [12]:
q9 = run_query("""
SELECT * FROM customers
WHERE strftime('%Y', join_date) = '2024' AND state = 'Maharashtra'
""")
q9


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1


**Q10. Find all orders placed between '2024-08-10' and '2024-08-25' (inclusive) that are NOT cancelled.**

In [13]:
q10 = run_query("""
SELECT * FROM orders
WHERE order_date BETWEEN '2024-08-10' AND '2024-08-25'
  AND status != 'Cancelled'
""")
q10


,order_id,customer_id,order_date,status,total_amount
0,1004,101,2024-08-10,Delivered,3499
1,1006,105,2024-08-15,Delivered,5898
2,1007,106,2024-08-18,Pending,1299
3,1008,103,2024-08-20,Delivered,899
4,1009,107,2024-08-25,Shipped,6098


**Q11. Explain what the index `idx_orders_date` does. How does it improve performance? Sample query that benefits from it.**

`idx_orders_date` is a B-Tree index built on the `order_date` column of `orders`. Without it, a query filtering by `order_date` has to do a **full table scan** — checking every single row one by one. With the index, the database keeps `order_date` values pre-sorted in a separate structure, so it can jump almost directly to the matching rows using a fast lookup (similar to using the index at the back of a book instead of reading every page). This turns an O(n) scan into something close to O(log n), which matters a lot as the `orders` table grows to millions of rows.

A query that directly benefits (range filter on `order_date`, exactly like Q10):


In [14]:
benefit_query = run_query("""
SELECT order_id, order_date, status
FROM orders
WHERE order_date BETWEEN '2024-08-01' AND '2024-08-15'
""")
benefit_query


,order_id,order_date,status
0,1001,2024-08-01,Delivered
1,1002,2024-08-03,Delivered
2,1003,2024-08-05,Shipped
3,1004,2024-08-10,Delivered
4,1005,2024-08-12,Cancelled
5,1006,2024-08-15,Delivered


**Q12. `SELECT * FROM customers WHERE YEAR(join_date) = 2024;` — would the index on `join_date` be used? Rewrite to be SARGable.**

No — there is no index on `join_date` in this schema in the first place (the indexes given are only on `city` and `state`). But even if one existed, wrapping the column in a function like `YEAR(join_date)` makes the query **non-SARGable**: the database has to *compute* `YEAR(join_date)` for every row before it can compare it to 2024, so it can't use a plain index seek on `join_date` — it forces a full scan regardless of whether an index exists.

The fix is to filter using a **range** on the raw column instead of transforming it, so any index on `join_date` could be used directly:


In [15]:
# SARGable version — works in SQLite too (date stored as 'YYYY-MM-DD' text)
sargable = run_query("""
SELECT * FROM customers
WHERE join_date >= '2024-01-01' AND join_date < '2025-01-01'
""")
sargable


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


---
## Section C — Aggregation (GROUP BY, SUM, COUNT, AVG, MIN, MAX)
These questions test the ability to summarize and aggregate data.


**Q13. Count the total number of orders.**

In [16]:
q13 = run_query("SELECT COUNT(*) AS total_orders FROM orders")
q13


,total_orders
0,10


**Q14. Find the total revenue (SUM of total_amount) from all 'Delivered' orders.**

In [17]:
q14 = run_query("""
SELECT SUM(total_amount) AS total_revenue_delivered
FROM orders WHERE status = 'Delivered'
""")
q14


,total_revenue_delivered
0,17191


**Q15. Calculate the average unit_price of products in each category.**

In [18]:
q15 = run_query("""
SELECT category, ROUND(AVG(unit_price), 2) AS avg_unit_price
FROM products
GROUP BY category
""")
q15


,category,avg_unit_price
0,Clothing,2699.0
1,Electronics,2224.0
2,Home,949.0


**Q16. For each order status, find the count of orders and the total revenue. Sort by total revenue descending.**

In [19]:
q16 = run_query("""
SELECT status,
       COUNT(*) AS order_count,
       SUM(total_amount) AS total_revenue
FROM orders
GROUP BY status
ORDER BY total_revenue DESC
""")
q16


,status,order_count,total_revenue
0,Delivered,6,17191
1,Shipped,2,13596
2,Cancelled,1,2999
3,Pending,1,1299


**Q17. Find the most expensive (MAX) and cheapest (MIN) product price in each category.**

In [20]:
q17 = run_query("""
SELECT category,
       MAX(unit_price) AS most_expensive,
       MIN(unit_price) AS cheapest
FROM products
GROUP BY category
""")
q17


,category,most_expensive,cheapest
0,Clothing,4599,799
1,Electronics,3499,899
2,Home,1299,599


**Q18. List all product categories where the average unit_price is greater than ₹2000. (Use HAVING)**

In [21]:
q18 = run_query("""
SELECT category, ROUND(AVG(unit_price), 2) AS avg_unit_price
FROM products
GROUP BY category
HAVING AVG(unit_price) > 2000
""")
q18


,category,avg_unit_price
0,Clothing,2699.0
1,Electronics,2224.0


---
## Section D — Joins & Relationships
These questions test the ability to combine data from multiple tables using different types of JOINs.


**Q19. INNER JOIN — each order with the customer's first_name and last_name. Show: order_id, order_date, first_name, last_name, total_amount.**

In [22]:
q19 = run_query("""
SELECT o.order_id, o.order_date, c.first_name, c.last_name, o.total_amount
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
""")
q19


,order_id,order_date,first_name,last_name,total_amount
0,1001,2024-08-01,Aarav,Sharma,4498
1,1002,2024-08-03,Priya,Patel,799
2,1003,2024-08-05,Rohan,Gupta,7498
3,1004,2024-08-10,Aarav,Sharma,3499
4,1005,2024-08-12,Sneha,Reddy,2999
5,1006,2024-08-15,Vikram,Singh,5898
6,1007,2024-08-18,Ananya,Iyer,1299
7,1008,2024-08-20,Rohan,Gupta,899
8,1009,2024-08-25,Karan,Mehta,6098
9,1010,2024-08-28,Divya,Nair,1598


**Q20. LEFT JOIN — ALL customers and their orders (if any). Customers with no orders should still appear with NULLs.**

In [23]:
q20 = run_query("""
SELECT c.customer_id, c.first_name, c.last_name, o.order_id, o.order_date, o.total_amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
""")
q20


,customer_id,first_name,last_name,order_id,order_date,total_amount
0,101,Aarav,Sharma,1001,2024-08-01,4498
1,101,Aarav,Sharma,1004,2024-08-10,3499
2,102,Priya,Patel,1002,2024-08-03,799
3,103,Rohan,Gupta,1003,2024-08-05,7498
4,103,Rohan,Gupta,1008,2024-08-20,899
5,104,Sneha,Reddy,1005,2024-08-12,2999
6,105,Vikram,Singh,1006,2024-08-15,5898
7,106,Ananya,Iyer,1007,2024-08-18,1299
8,107,Karan,Mehta,1009,2024-08-25,6098
9,108,Divya,Nair,1010,2024-08-28,1598


**Q21. 3-table JOIN (orders → order_items → products): order_id, product_name, quantity, unit_price, discount_pct.**

In [24]:
q21 = run_query("""
SELECT oi.order_id, p.product_name, oi.quantity, oi.unit_price, oi.discount_pct
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
ORDER BY oi.order_id
""")
q21


,order_id,product_name,quantity,unit_price,discount_pct
0,1001,Wireless Earbuds,2,1499,0
1,1001,Laptop Stand,1,899,10
2,1002,Cotton T-Shirt,1,799,0
3,1003,Smart Watch,1,2999,0
4,1003,Running Shoes,1,4599,5
5,1004,Bluetooth Speaker,1,3499,0
6,1005,Smart Watch,1,2999,0
7,1006,Wireless Earbuds,1,1499,10
8,1006,Running Shoes,1,4599,5
9,1007,Bedsheet Set,1,1299,0


**Q22. Difference between LEFT JOIN and RIGHT JOIN, with an example from this schema. When to use FULL OUTER JOIN?**

- **LEFT JOIN** keeps *every row from the left (first) table*, and brings in matching rows from the right table — filling in NULLs where there's no match. Example: `customers LEFT JOIN orders` (Q20 above) — every customer shows up even if they've never placed an order.
- **RIGHT JOIN** does the mirror opposite: it keeps *every row from the right table*, filling NULLs for the left side where there's no match. The same idea written as a RIGHT JOIN would be `orders RIGHT JOIN customers` — every customer appears, with order columns NULL if they have none. (Note: SQLite added RIGHT JOIN support only in fairly recent versions — LEFT JOIN is the universally safe choice and is used throughout this notebook.)
- **FULL OUTER JOIN** keeps *all rows from both tables*, matched where possible, NULL on whichever side doesn't have a match. You'd reach for it when you need a complete picture from both directions in one result — e.g. "show every customer AND every order, even ones with mismatched/orphaned customer_ids," so nothing from either table is silently dropped.


**Q23. Identify all Foreign Key relationships. What happens inserting an order with customer_id = 999 (doesn't exist)?**

**Foreign Key relationships in this schema:**
- `orders.customer_id` → `customers.customer_id`
- `order_items.order_id` → `orders.order_id`
- `order_items.product_id` → `products.product_id`

Inserting an order with a `customer_id` that doesn't exist in `customers` violates referential integrity. Since we turned on `PRAGMA foreign_keys = ON` at the start of this notebook, SQLite will **reject** the insert with a `FOREIGN KEY constraint failed` error. Demonstrated below:


In [25]:
try:
    cursor.execute(
        "INSERT INTO orders VALUES (9999, 999, '2024-09-01', 'Pending', 1000.00)"
    )
    conn.commit()
except sqlite3.IntegrityError as e:
    print("INSERT FAILED as expected.")
    print("Error:", e)
    conn.rollback()


INSERT FAILED as expected.
Error: FOREIGN KEY constraint failed


---
## Section E — Advanced Concepts (CASE, ACID, Transactions)
These questions test understanding of conditional logic, database reliability principles, and transaction management.


**Q24. CASE to classify products into price tiers:**
- 'Budget' → unit_price < 1000
- 'Mid-Range' → 1000–3000
- 'Premium' → unit_price > 3000


In [26]:
q24 = run_query("""
SELECT product_name, unit_price,
       CASE
           WHEN unit_price < 1000 THEN 'Budget'
           WHEN unit_price BETWEEN 1000 AND 3000 THEN 'Mid-Range'
           ELSE 'Premium'
       END AS price_tier
FROM products
""")
q24


,product_name,unit_price,price_tier
0,Wireless Earbuds,1499,Mid-Range
1,Cotton T-Shirt,799,Budget
2,Smart Watch,2999,Mid-Range
3,Running Shoes,4599,Premium
4,Bluetooth Speaker,3499,Premium
5,Bedsheet Set,1299,Mid-Range
6,Laptop Stand,899,Budget
7,Cushion Covers (Set),599,Budget


**Q25. CASE inside an aggregate — count 'Delivered' vs 'Not Delivered' orders in a single row.**

In [27]:
q25 = run_query("""
SELECT
    SUM(CASE WHEN status = 'Delivered' THEN 1 ELSE 0 END) AS delivered_count,
    SUM(CASE WHEN status != 'Delivered' THEN 1 ELSE 0 END) AS not_delivered_count
FROM orders
""")
q25


,delivered_count,not_delivered_count
0,6,4


**Q26. Explain each letter of ACID with a real-world bank-transfer example.**

Imagine transferring ₹5000 from Account A to Account B. That single transfer is really *two* database operations: debit A, credit B.

- **A — Atomicity:** Both steps happen together, or neither does. If the debit from A succeeds but the system crashes before crediting B, atomicity guarantees the *whole* transaction rolls back — money can't vanish into thin air mid-transfer.
- **C — Consistency:** The database must move from one valid state to another valid state. Total money in the bank before and after the transfer must match — the transaction can't leave the books in an inconsistent state (e.g. debiting A without crediting B anywhere).
- **I — Isolation:** If two transfers happen at the same time (e.g. someone else is also reading Account A's balance), each transaction should behave as if it's running alone — one shouldn't see the other's half-finished changes (like a balance that's been debited but not yet credited anywhere).
- **D — Durability:** Once the transfer is committed and the bank shows "Transfer Successful," that result must survive even a power failure or crash one second later — the change is permanently saved to disk, not just sitting in memory.

Without ACID, a bank transfer could lose money, double-credit it, show a wrong intermediate balance to a concurrent reader, or even forget a successful transfer ever happened.


**Q27. Full transaction — insert order 1011 + 2 order items + update stock, with ROLLBACK on failure / COMMIT on success.**

In [28]:
from datetime import date

today = date.today().isoformat()

try:
    cursor.execute("BEGIN")

    # 1. Insert the new order
    cursor.execute(
        "INSERT INTO orders VALUES (1011, 102, ?, 'Pending', 1598.00)",
        (today,)
    )

    # 2. Insert two order items for that order (e.g. one Bedsheet Set + one Cushion Covers set)
    cursor.execute(
        "INSERT INTO order_items VALUES (5016, 1011, 206, 1, 1299.00, 0)"
    )
    cursor.execute(
        "INSERT INTO order_items VALUES (5017, 1011, 208, 1, 599.00, 0)"
    )

    # 3. Update stock_qty of the purchased products
    cursor.execute(
        "UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 206"
    )
    cursor.execute(
        "UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 208"
    )

    # 4. All steps succeeded -> COMMIT
    conn.commit()
    print("Transaction COMMITTED successfully.")

except sqlite3.Error as e:
    # Any failure anywhere above -> ROLLBACK the entire transaction
    conn.rollback()
    print("Transaction FAILED, ROLLED BACK. Error:", e)

# Verify: order 1011 and updated stock
display(run_query("SELECT * FROM orders WHERE order_id = 1011"))
display(run_query("SELECT * FROM order_items WHERE order_id = 1011"))
display(run_query("SELECT product_id, product_name, stock_qty FROM products WHERE product_id IN (206, 208)"))


Transaction COMMITTED successfully.


,order_id,customer_id,order_date,status,total_amount
0,1011,102,2026-06-26,Pending,1598


,item_id,order_id,product_id,quantity,unit_price,discount_pct
0,5016,1011,206,1,1299,0
1,5017,1011,208,1,599,0


,product_id,product_name,stock_qty
0,206,Bedsheet Set,299
1,208,Cushion Covers (Set),399


---
## Conclusion

All 27 questions across Sections A–E have been answered with executable SQL queries, their live results, and explanations where required (Primary Keys, constraints, indexes, JOIN types, ACID, and a full atomic transaction).

**Tools used:** Python `sqlite3` (standard SQL syntax, as permitted by the assignment) + `pandas` for clean result display.
